# Example 4: Tool Calling —— Function Calling

演示 Hy3 的 tool calling（Function Calling）能力：
- **单次调用**：模型识别需调用 tool，返回函数名和参数
- **多轮工具循环**：模型 → tool → 模型 → tool → ... → 最终回答

**前置条件：**
1. 在 [TokenHub 控制台](https://console.cloud.tencent.com/tokenhub/apikey) 创建 API Key
2. 安装 openai: `pip install openai`

In [ ]:
import json
from openai import OpenAI

API_KEY = "sk-你的APIKey"  # TODO: 替换

client = OpenAI(
    base_url="https://tokenhub.tencentmaas.com/v1",
    api_key=API_KEY,
)

# 定义两个 tool
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "获取指定城市的当前天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "城市名称"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "温度单位"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "执行数学计算",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "数学表达式"},
                },
                "required": ["expression"],
            },
        },
    },
]


def execute_tool(tool_call):
    """模拟执行 tool 调用"""
    fn_name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    if fn_name == "get_weather":
        weather = {"北京": {"temperature": 28, "condition": "晴"},
                   "上海": {"temperature": 32, "condition": "多云"},
                   "深圳": {"temperature": 30, "condition": "阵雨"}}
        info = weather.get(args["city"], {"temperature": 25, "condition": "未知"})
        return json.dumps(info)
    elif fn_name == "calculate":
        try:
            result = eval(args["expression"], {"__builtins__": {}}, {})
            return json.dumps({"expression": args["expression"], "result": result})
        except Exception as e:
            return json.dumps({"error": str(e)})
    return json.dumps({"error": f"Unknown tool: {fn_name}"})

## 1. 单次 Tool Calling

In [ ]:
print("单次 Tool Calling\n")

response = client.chat.completions.create(
    model="hy3",
    messages=[{"role": "user", "content": "北京今天天气怎么样？"}],
    temperature=0.7, top_p=1.0, tools=tools, tool_choice="auto",
    extra_body={"reasoning_effort": "no_think"},
)

message = response.choices[0].message
if message.tool_calls:
    for tc in message.tool_calls:
        print(f"调用 tool: {tc.function.name}")
        print(f"参数: {tc.function.arguments}")
        result = execute_tool(tc)
        print(f"结果: {result}\n")
else:
    print(f"直接回答: {message.content}")

## 2. 多轮 Tool Calling 循环

模型调 tool → 执行 → 返回结果 → 模型综合回答

In [ ]:
print("多轮 Tool Calling 循环\n")

messages = [
    {"role": "system", "content": "你是一个有用的助手，可以使用 tool 来帮助用户。"},
    {"role": "user", "content": "帮我计算 12345 × 6789 的结果，然后查一下深圳的天气，最后用中文总结。"},
]

MAX_TURNS = 5
for turn in range(MAX_TURNS):
    print(f"--- 第 {turn + 1} 轮 ---")
    response = client.chat.completions.create(
        model="hy3", messages=messages, temperature=0.7, top_p=1.0,
        tools=tools, tool_choice="auto",
        extra_body={"reasoning_effort": "no_think"},
    )
    message = response.choices[0].message

    if message.tool_calls:
        # 将 tool_calls 加入 messages
        messages.append({
            "role": "assistant", "content": message.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in message.tool_calls
            ],
        })
        for tc in message.tool_calls:
            print(f"  🔧 {tc.function.name}({tc.function.arguments})")
            result = execute_tool(tc)
            print(f"  ✅ {result}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    else:
        print(f"\n  💬 最终回答:\n  {message.content}")
        break
else:
    print(f"\n  ⚠️ 达到最大轮数 {MAX_TURNS}")

print(f"\n共 {len(messages)} 条消息")